In [ ]:
from transformers import AutoModelForSequenceClassification,AutoTokenizer
import torch

In [ ]:
model_name = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'

model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [74]:
user_claim = "The sky is green."
top_evidence_from_reranker = "Rayleigh scattering of sunlight in Earth's atmosphere causes diffuse sky radiation, which is why the sky appears ."

input = tokenizer(top_evidence_from_reranker,user_claim,padding = True,truncation=True,return_tensors='pt')

with torch.no_grad():
    outputs= model(**input)


In [38]:
outputs.logits
probs = torch.softmax(outputs.logits[0],dim = -1)
probs
preds = torch.argmax(probs)
label_names = ["entailment", "neutral", "contradiction"]

label_names[preds.item()]

'contradiction'

In [60]:
input

{'input_ids': tensor([[     1, 102959,  25864,    265,   9522,    267,   2610,    280,    268,
           3384,   2884,  25379,   3907,   6681,    261,    319,    269,    579,
            262,   3907,   2194,   1707,    260,      2,    279,   3907,    269,
           1509,    260,      2]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])}

In [ ]:
def better_classifier(claim,top_evidence):
    

In [ ]:
# class classifier_ugly:
#     def __init__(self):
#         self.model_name = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
#         self.label_names = ["entailment", "neutral", "contradiction"]
#         self.verdicts = []
        # self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


#         try:
#             self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
#             self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            # self.model.to(self.device)
# 
#         except Exception as e:
#             print(f"Could not fetch model from Hugging Face | {e} ")

#     def classify(self,claim,top_evidence):
#         # top evidence should be a list of paired tuples where second element of each tuple is text of reranked evidence
#         self.verdicts = [] # resetting the verdict list
#         evidences = [evidence[1] for evidence in top_evidence]
#         if not evidences:
#             print("Could not fetch evidences from top evidence")
#             return
#         try:
#             claim = str(claim)
#             for evidence in evidences:
#                 inputs = self.tokenizer(evidence,claim,return_tensors='pt',padding=True)
#                 if not inputs:
#                     print(f"Could not tokenize the inputs for evidence {evidence}")
#                     continue

#                 with torch.no_grad():

                #inputs = {k:v.to(self.device) for k,v in inputs.items()}
# 
#                     outputs = self.model(**inputs)

#                 probs  = torch.softmax(outputs.logits[0],dim=-1)
#                 pred = torch.argmax(probs).item()
#                 self.verdicts.append({
#                     'evidence':evidence,
#                     'verdict':self.label_names[pred],
#                     'scores': [{name:float(probs[i].item())} for i,name in enumerate(self.label_names)]
#                 })

#             labels = [v['verdict'] for v in self.verdicts]
#             if 'entailment' in labels:
#                 result = "TRUE"
#             elif "contradiction" in labels:
#                 result = "FALSE"
#             else:
#                 result = "NEUTRAL"
#         except Exception as e:
#             print(f"Could not run the model | {e}")

#         return result,self.verdicts
        

In [77]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

class Classifier:
    def __init__(self):
        self.model_name = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
        self.label_names = ["entailment", "neutral", "contradiction"]
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(self.device)


        try:
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model.to(self.device)

        except Exception as e:
            raise RuntimeError(f"Could not fetch model from Hugging Face | {e}")

    def classify(self, claim, top_evidence):
        
        self.verdicts = []  #
        evidences = [e[1] for e in top_evidence]

        if not evidences:
            raise ValueError("No evidence provided")

        try:
            inputs = self.tokenizer(
                evidences,
                [claim] * len(evidences),
                return_tensors="pt",
                padding=True,
                truncation=True
            )

            with torch.no_grad():
                inputs = {k:v.to(self.device) for k,v in inputs.items()}
                outputs = self.model(**inputs)

            probs = torch.softmax(outputs.logits, dim=-1)

            for i, evidence in enumerate(evidences):
                pred = torch.argmax(probs[i]).item()
                self.verdicts.append({
                    "evidence": evidence,
                    "verdict": self.label_names[pred],
                    "scores": {name: float(probs[i][j]) for j, name in enumerate(self.label_names)}
                })

            labels = [v["verdict"] for v in self.verdicts]
            if "entailment" in labels:
                result = "TRUE"
            elif "contradiction" in labels:
                result = "FALSE"
            else:
                result = "NEUTRAL"

            return result, self.verdicts

        except Exception as e:
            raise RuntimeError(f"Classification failed | {e}")
        
    def __call__(self,claim,evidences):
        return self.classify(claim,evidences)


In [78]:
yoho = Classifier()

cuda


In [81]:
user_claim = "The sky is green."
top_evidence_from_reranker = [(0,"Rayleigh scattering of sunlight in Earth's atmosphere causes diffuse sky radiation, which is why the sky appears blue.")]


result , verdicts = yoho(user_claim,top_evidence_from_reranker)

In [82]:
result,verdicts

('FALSE',
 [{'evidence': "Rayleigh scattering of sunlight in Earth's atmosphere causes diffuse sky radiation, which is why the sky appears blue.",
   'verdict': 'contradiction',
   'scores': {'entailment': 0.0001099365035770461,
    'neutral': 0.0008646054193377495,
    'contradiction': 0.9990254640579224}}])